# CLIP Linear Projection to Image Tokens

## Pipeline:
*CLIP features [B, clip_dim] -> Linear(clip_dim, clip_length * embedding_dim) -> projected features [B, clip_length * embedding_dim] -> reshape -> image tokens [B, clip_length, embedding_dim]*

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import torch
from torch import Tensor, nn
from transformers import AutoConfig

import config as project_config

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate project config.py")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

c:\Users\ADMIN\miniconda3\envs\zfs-caption\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.11.0+cu128
CUDA available: True


## Interface cần thống nhất

- Input: floating-point tensor có shape `[batch_size, clip_dim]`.
- Output: floating-point tensor có shape `[batch_size, clip_length, embedding_dim]`.
- `clip_dim` được lấy từ `CLIPConfig.projection_dim`.
- `embedding_dim` được lấy từ config của language model qua thuộc tính chuẩn `hidden_size`.
- `clip_length=10` là hyperparameter riêng của Mapping Network, không thuộc config CLIP hoặc GPT-2.
- Linear projection không dùng activation, normalization hoặc vòng lặp.
- `clip_length` và `embedding_dim` là tham số để TV3 có thể thay đổi cấu hình mà không sửa `forward`.

In [2]:
class ClipProjection(nn.Module):
    """Project one global CLIP feature into a sequence of image tokens."""

    def __init__(
        self,
        clip_dim: int,
        embedding_dim: int,
        clip_length: int,
    ) -> None:
        super().__init__()

        dimensions = {
            "clip_dim": clip_dim,
            "embedding_dim": embedding_dim,
            "clip_length": clip_length,
        }
        for name, value in dimensions.items():
            if isinstance(value, bool) or not isinstance(value, int) or value < 1:
                raise ValueError(f"{name} must be a positive integer")

        self.clip_dim = clip_dim
        self.embedding_dim = embedding_dim
        self.clip_length = clip_length
        self.projection = nn.Linear(
            in_features=clip_dim,
            out_features=clip_length * embedding_dim,
        )

    def forward(self, clip_features: Tensor) -> Tensor:
        if clip_features.ndim != 2:
            raise ValueError(
                "clip_features must have shape [batch_size, clip_dim]"
            )
        if clip_features.shape[1] != self.clip_dim:
            raise ValueError(
                "Unexpected CLIP feature dimension: "
                f"{clip_features.shape[1]} != {self.clip_dim}"
            )

        batch_size = clip_features.shape[0]
        projected_features = self.projection(clip_features)
        return projected_features.reshape(
            batch_size,
            self.clip_length,
            self.embedding_dim,
        )

## Giải thích phép biến đổi

Với mỗi CLIP feature `x`, Linear layer tính `y = Wx + b` và tạo ra `clip_length * embedding_dim` giá trị. `reshape` chỉ thay đổi cách nhìn tensor thành một chuỗi image token; nó không thay đổi hoặc sao chép giá trị.

```text
Input một sample:  [clip_dim]
Projection output: [clip_length * embedding_dim]
Image tokens:      [clip_length, embedding_dim]
```

Mỗi đoạn `embedding_dim` phần tử của projection output trở thành một image token. Các token này là learned views của cùng global CLIP feature, không phải các patch không gian thật của ảnh.

In [3]:
clip_model_config = AutoConfig.from_pretrained(
    project_config.CLIP_MODEL_NAME
)
language_model_config = AutoConfig.from_pretrained(
    project_config.GPT2_MODEL_NAME
)

CLIP_DIM = int(clip_model_config.projection_dim)
EMBEDDING_DIM = int(language_model_config.hidden_size)
CLIP_LENGTH = 10

torch.manual_seed(project_config.SEED)
clip_projection = ClipProjection(
    clip_dim=CLIP_DIM,
    embedding_dim=EMBEDDING_DIM,
    clip_length=CLIP_LENGTH,
)

parameter_count = sum(
    parameter.numel()
    for parameter in clip_projection.parameters()
)
expected_parameter_count = (
    CLIP_DIM * CLIP_LENGTH * EMBEDDING_DIM
    + CLIP_LENGTH * EMBEDDING_DIM
)

assert parameter_count == expected_parameter_count
print("CLIP model         :", project_config.CLIP_MODEL_NAME)
print("Language model     :", project_config.GPT2_MODEL_NAME)
print("CLIP feature dim   :", CLIP_DIM)
print("Token embedding dim:", EMBEDDING_DIM)
print("Image-token length :", CLIP_LENGTH)
print(f"Projection parameters: {parameter_count:,}")

CLIP model         : openai/clip-vit-base-patch32
Language model     : openai-community/gpt2
CLIP feature dim   : 512
Token embedding dim: 768
Image-token length : 10
Projection parameters: 3,939,840


Số tham số được tính động từ model config theo công thức `clip_dim * clip_length * embedding_dim + clip_length * embedding_dim`. Đây là toàn bộ trainable parameters trong phần TV1.

In [4]:
def check_output_shape(batch_size: int) -> None:
    clip_features = torch.randn(batch_size, CLIP_DIM)
    image_tokens = clip_projection(clip_features)

    expected_shape = (batch_size, CLIP_LENGTH, EMBEDDING_DIM)
    assert image_tokens.shape == expected_shape
    assert image_tokens.dtype == clip_features.dtype
    assert image_tokens.device == clip_features.device
    assert torch.isfinite(image_tokens).all()

    print(
        f"Batch {batch_size}: "
        f"{tuple(clip_features.shape)} -> {tuple(image_tokens.shape)}"
    )


for test_batch_size in (1, 4, 32):
    check_output_shape(test_batch_size)

alternative_projection = ClipProjection(
    clip_dim=7,
    embedding_dim=12,
    clip_length=3,
)
alternative_output = alternative_projection(torch.randn(2, 7))
assert alternative_output.shape == (2, 3, 12)
print("Alternative dimensions: (2, 7) ->", tuple(alternative_output.shape))

print("Shape, dtype, device and finite-value checks passed")

Batch 1: (1, 512) -> (1, 10, 768)
Batch 4: (4, 512) -> (4, 10, 768)
Batch 32: (32, 512) -> (32, 10, 768)
Alternative dimensions: (2, 7) -> (2, 3, 12)
Shape, dtype, device and finite-value checks passed


In [5]:
clip_projection.zero_grad(set_to_none=True)
gradient_input = torch.randn(4, CLIP_DIM, requires_grad=True)
gradient_output = clip_projection(gradient_input)
position_weights = torch.linspace(
    0.5,
    1.5,
    EMBEDDING_DIM,
).reshape(1, 1, EMBEDDING_DIM)
loss = (gradient_output * position_weights).mean()
loss.backward()

gradients = {
    "input": gradient_input.grad,
    "projection weight": clip_projection.projection.weight.grad,
    "projection bias": clip_projection.projection.bias.grad,
}
for name, gradient in gradients.items():
    assert gradient is not None, f"Missing gradient: {name}"
    assert torch.isfinite(gradient).all(), f"Invalid gradient: {name}"
    assert gradient.abs().sum() > 0, f"Zero gradient: {name}"

print("Gradient checks passed")

Gradient checks passed


In [6]:
invalid_inputs = (
    torch.randn(4, CLIP_DIM - 1),
    torch.randn(4, CLIP_DIM + 1),
    torch.randn(4, 1, CLIP_DIM),
)

for invalid_input in invalid_inputs:
    try:
        clip_projection(invalid_input)
    except ValueError as error:
        print(f"Rejected shape {tuple(invalid_input.shape)}: {error}")
    else:
        raise AssertionError(
            f"Invalid shape was accepted: {tuple(invalid_input.shape)}"
        )

base_configuration = {
    "clip_dim": CLIP_DIM,
    "embedding_dim": EMBEDDING_DIM,
    "clip_length": CLIP_LENGTH,
}
invalid_configurations = (
    {"clip_dim": 0},
    {"embedding_dim": -1},
    {"clip_length": 0},
    {"clip_length": True},
)

for invalid_configuration in invalid_configurations:
    configuration = {**base_configuration, **invalid_configuration}
    try:
        ClipProjection(**configuration)
    except ValueError as error:
        print(f"Rejected config {invalid_configuration}: {error}")
    else:
        raise AssertionError(
            f"Invalid configuration was accepted: {invalid_configuration}"
        )

Rejected shape (4, 511): Unexpected CLIP feature dimension: 511 != 512
Rejected shape (4, 513): Unexpected CLIP feature dimension: 513 != 512
Rejected shape (4, 1, 512): clip_features must have shape [batch_size, clip_dim]
Rejected config {'clip_dim': 0}: clip_dim must be a positive integer
Rejected config {'embedding_dim': -1}: embedding_dim must be a positive integer
Rejected config {'clip_length': 0}: clip_length must be a positive integer
Rejected config {'clip_length': True}: clip_length must be a positive integer


## Kiểm tra tùy chọn với CLIP feature thật

Cell sau không tạo hoặc tải CLIP model. Nó chỉ đọc cache `clip_features.pt` nếu file đã tồn tại. Nếu chưa có cache, cell sẽ bỏ qua để notebook vẫn dùng được trong quá trình review interface.

In [7]:
feature_path = project_config.FEATURE_DIR / "clip_features.pt"

if feature_path.is_file():
    clip_feature_data = torch.load(feature_path, map_location="cpu")
    real_features = clip_feature_data["features"][:4]
    cached_model_name = clip_feature_data.get("clip_model")
    cached_feature_dim = int(
        clip_feature_data.get("feature_dim", real_features.shape[1])
    )

    if cached_model_name != project_config.CLIP_MODEL_NAME:
        raise ValueError(
            "CLIP feature cache was created by a different model: "
            f"{cached_model_name!r} != {project_config.CLIP_MODEL_NAME!r}. "
            "Regenerate clip_features.pt before training."
        )
    if cached_feature_dim != CLIP_DIM:
        raise ValueError(
            "Cached CLIP feature dimension does not match model config: "
            f"{cached_feature_dim} != {CLIP_DIM}. "
            "Regenerate clip_features.pt before training."
        )

    if real_features.ndim != 2 or real_features.shape[1] != CLIP_DIM:
        raise ValueError(
            f"Cached CLIP features must have shape [N, {CLIP_DIM}], got "
            f"{tuple(real_features.shape)}"
        )

    clip_projection.eval()
    with torch.no_grad():
        real_image_tokens = clip_projection(real_features)

    assert real_image_tokens.shape == (
        real_features.shape[0],
        CLIP_LENGTH,
        EMBEDDING_DIM,
    )
    assert torch.isfinite(real_image_tokens).all()

    print("Real CLIP input shape :", tuple(real_features.shape))
    print("Real image-token shape:", tuple(real_image_tokens.shape))
else:
    print(f"Skipped: CLIP feature cache not found at {feature_path.resolve()}")

Real CLIP input shape : (4, 512)
Real image-token shape: (4, 10, 768)
